# 1.content的使用

举例1：保存多模态的数据,使用字典列表

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

MODEL_NAME = "deepseek-flash"

model = init_chat_model(
    model=MODEL_NAME,
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

def encode_image(img_path, img_type="jpeg"):
    """将本地图片转换为 Base64 编码的 Data URI 字符串"""
    with open(img_path, "rb") as img_file:
        b64 = base64.b64encode(img_file.read()).decode("utf-8")
    return f"data:image/{img_type};base64,{b64}"

img_path = "image_test.png"
base64_image = encode_image(img_path, img_type="png")   # 注意类型一致

res = model.invoke(
    [
        HumanMessage(
            content=[
                {"type": "text", "text": "这张图里面有什么？"},
                {
                    "type": "image_url",
                    "image_url": {"url": base64_image},   # ← 用字典包一层
                },
            ]
        )
    ]
)

print(res.content)

# 2.content_blocks的使用

举例1：输入的格式化

In [ ]:
import base64
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

MODEL_NAME = "deepseek-flash"

model = init_chat_model(
    model=MODEL_NAME,
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串，方便在文本嵌入图片数据 """
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

res = model.invoke(
    [
        HumanMessage(
        content_blocks=[
            {"type": "text", "text": "这张图里面有什么？"},
                {
                    "type": "image",
                    "base64": base64_image,
                    "mime_type": "image/png",
                },
            ]
        )
    ]
)

print(res.content)

举例2：输出的格式化

In [7]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from  rich import print as rprint

load_dotenv()

model = init_chat_model(
    model = "deepseek:deepseek-flash",
    extra_body = {"thinking": {"type": "enabled"}},   # 思考模式
)

res = model.invoke("你好，一句话回答")

rprint('=' * 20, '-> res <-', '=' * 20)
rprint(res)
rprint('=' * 20, '-> res.content <-', '=' * 20)
rprint(res.content)
rprint('=' * 20, '-> res.content_blocks <-', '=' * 20)
rprint(res.content_blocks)

==================== -> res <- ====================

AIMessage(
    content='你好，请问有什么可以帮你？',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们需要回答用户。用户说“你好，一句话回答”。我们需要一句回应。需要遵守？用中文一句话回答。可以简单问候。注意必须一
句话。可以“你好，很高兴为你服务，请问有什么可以帮你？” 这是一句话。应该可以。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 60,
            'prompt_tokens': 34,
            'total_tokens': 94,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 52,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 34
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '16a3b49a-3269-44cb-91b7-984e643cb816',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a125eb-488e-7153-9b41-f2dda8f1b733-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 34,
        'output_tokens': 60,
        'total_tokens': 94,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 52}
    }
)

==================== -> res.content <- ====================

你好，请问有什么可以帮你？

==================== -> res.content_blocks <- ====================

[
    {
        'type': 'reasoning',
        'reasoning': 
'我们需要回答用户。用户说“你好，一句话回答”。我们需要一句回应。需要遵守？用中文一句话回答。可以简单问候。注意必须一
句话。可以“你好，很高兴为你服务，请问有什么可以帮你？” 这是一句话。应该可以。'
    },
    {'type': 'text', 'text': '你好，请问有什么可以帮你？'}
]